# 2.8 — RAG Evaluation

How do you know if your RAG pipeline is actually working well?

We measure quality across 4 dimensions:

| Metric | Question it answers |
|--------|--------------------|
| **Faithfulness** | Is the answer supported by the retrieved context? (no hallucination) |
| **Answer Relevance** | Does the answer actually address the question? |
| **Context Recall** | Did we retrieve the right chunks? |
| **Context Precision** | Are the retrieved chunks actually useful? |

We'll use two approaches:
1. **LLM-as-Judge** — use the LLM to score each metric (works locally with Ollama)
2. **RAGAS** — popular open-source RAG evaluation framework

In [2]:
!pip install langchain langchain-ollama langchain-community chromadb --quiet

## Step 1 — Build the RAG Pipeline (same as 2.5)

In [3]:
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Documents
docs = [
    Document(page_content='All full-time employees receive 20 days of annual leave per year.'),
    Document(page_content='Sick leave is up to 10 days per year with a medical certificate.'),
    Document(page_content='Parental leave is 16 weeks fully paid for primary caregivers.'),
    Document(page_content='Leave requests must be submitted at least 2 weeks in advance.'),
    Document(page_content='Employees may work remotely up to 3 days per week.'),
    Document(page_content='Remote workers must be available during core hours: 10am to 3pm.'),
    Document(page_content='Health insurance is provided for all full-time employees and their immediate family.'),
    Document(page_content='A gym membership subsidy of $50 per month is available.'),
    Document(page_content='Employees receive a $1,000 annual learning and development budget.'),
    Document(page_content='Standard working hours are 9am to 5pm, Monday to Friday.'),
    Document(page_content='Overtime must be pre-approved and compensated at 1.5x the hourly rate.'),
]

# Vector store
embeddings = OllamaEmbeddings(model='llama3.1')
vectorstore = Chroma.from_documents(docs, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

# LLM
llm = ChatOllama(model='llama3.1', temperature=0)

# RAG chain
prompt = ChatPromptTemplate.from_template("""
You are a helpful HR assistant. Answer using only the context below.
If the answer is not in the context, say "I don't have that information."

Context:
{context}

Question: {question}
Answer:
""")

def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

print('RAG pipeline ready.')

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


RAG pipeline ready.


## Step 2 — Create a Test Dataset

A test dataset contains:
- **question** — the user query
- **ground_truth** — the correct answer (what we expect)
- **ground_truth_context** — which document chunk should be retrieved

In [4]:
test_dataset = [
    {
        'question': 'How many annual leave days do employees get?',
        'ground_truth': '20 days per year',
        'ground_truth_context': 'All full-time employees receive 20 days of annual leave per year.'
    },
    {
        'question': 'How many sick leave days are allowed?',
        'ground_truth': '10 days per year with a medical certificate',
        'ground_truth_context': 'Sick leave is up to 10 days per year with a medical certificate.'
    },
    {
        'question': 'How many days can I work from home per week?',
        'ground_truth': '3 days per week',
        'ground_truth_context': 'Employees may work remotely up to 3 days per week.'
    },
    {
        'question': 'What is the overtime pay rate?',
        'ground_truth': '1.5x the hourly rate',
        'ground_truth_context': 'Overtime must be pre-approved and compensated at 1.5x the hourly rate.'
    },
    {
        'question': 'What is the company stock price?',
        'ground_truth': 'Not available in the document',
        'ground_truth_context': None  # no relevant chunk exists
    },
]

print(f'Test dataset: {len(test_dataset)} questions')

Test dataset: 5 questions


## Step 3 — Run the RAG Pipeline on All Test Questions

In [5]:
results = []

for item in test_dataset:
    question = item['question']

    # Get retrieved context
    retrieved_docs = retriever.invoke(question)
    retrieved_context = format_docs(retrieved_docs)

    # Get generated answer
    answer = rag_chain.invoke(question)

    results.append({
        'question':           question,
        'ground_truth':       item['ground_truth'],
        'ground_truth_context': item['ground_truth_context'],
        'retrieved_context':  retrieved_context,
        'answer':             answer,
    })
    print(f'Q: {question}')
    print(f'A: {answer[:80]}...' if len(answer) > 80 else f'A: {answer}')
    print()

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Q: How many annual leave days do employees get?
A: 20 days per year.

Q: How many sick leave days are allowed?
A: Up to 10 days per year with a medical certificate.

Q: How many days can I work from home per week?
A: Up to 3 days per week.

Q: What is the overtime pay rate?
A: 1.5x the hourly rate.

Q: What is the company stock price?
A: I don't have that information.



## Step 4 — LLM-as-Judge Evaluation

We use the LLM itself to score each metric. This works fully locally with Ollama — no API key needed.

In [6]:
judge_prompt = ChatPromptTemplate.from_template("""
You are an expert evaluator for RAG (Retrieval-Augmented Generation) systems.
Score the following on a scale of 0.0 to 1.0. Output ONLY a number, nothing else.

Metric: {metric}
Definition: {definition}

Question: {question}
Context: {context}
Answer: {answer}
Ground Truth: {ground_truth}

Score (0.0 to 1.0):
""")

judge_chain = judge_prompt | llm | StrOutputParser()

def score_metric(metric, definition, question, context, answer, ground_truth):
    response = judge_chain.invoke({
        'metric': metric,
        'definition': definition,
        'question': question,
        'context': context,
        'answer': answer,
        'ground_truth': ground_truth,
    })
    try:
        return round(float(response.strip()), 2)
    except ValueError:
        return 0.0

print('Judge ready.')

Judge ready.


In [7]:
metrics = [
    (
        'Faithfulness',
        'Is every claim in the answer supported by the context? Score 1.0 if fully supported, 0.0 if the answer contains facts not in the context.'
    ),
    (
        'Answer Relevance',
        'Does the answer directly address the question asked? Score 1.0 if fully relevant, 0.0 if the answer is off-topic or vague.'
    ),
    (
        'Context Recall',
        'Does the retrieved context contain all the information needed to answer the question correctly? Score 1.0 if yes, 0.0 if key information is missing.'
    ),
]

print(f'{"Question":<45} {"Faithful":>10} {"Relevance":>10} {"Recall":>8}')
print('-' * 78)

all_scores = []

for r in results:
    row_scores = {}
    for metric_name, metric_def in metrics:
        score = score_metric(
            metric=metric_name,
            definition=metric_def,
            question=r['question'],
            context=r['retrieved_context'],
            answer=r['answer'],
            ground_truth=r['ground_truth'],
        )
        row_scores[metric_name] = score

    all_scores.append(row_scores)
    q = r['question'][:43] + '..' if len(r['question']) > 43 else r['question']
    print(f'{q:<45} {row_scores["Faithfulness"]:>10} {row_scores["Answer Relevance"]:>10} {row_scores["Context Recall"]:>8}')

print('-' * 78)
avg_faith   = sum(s['Faithfulness']     for s in all_scores) / len(all_scores)
avg_rel     = sum(s['Answer Relevance'] for s in all_scores) / len(all_scores)
avg_recall  = sum(s['Context Recall']   for s in all_scores) / len(all_scores)
print(f'{"AVERAGE":<45} {avg_faith:>10.2f} {avg_rel:>10.2f} {avg_recall:>8.2f}')

Question                                        Faithful  Relevance   Recall
------------------------------------------------------------------------------
How many annual leave days do employees get..        1.0        1.0      1.0
How many sick leave days are allowed?                1.0        1.0      1.0
How many days can I work from home per week..        1.0        1.0      1.0
What is the overtime pay rate?                       1.0        1.0      1.0
What is the company stock price?                     0.5        0.5      0.0
------------------------------------------------------------------------------
AVERAGE                                             0.90       0.90     0.80


## Step 5 — Context Precision (Did We Retrieve the Right Chunk?)

Context Precision checks whether the **ground truth chunk** appeared in the retrieved results.

In [8]:
def context_precision(retrieved_context: str, ground_truth_context: str) -> float:
    """Check if the ground truth chunk is present in the retrieved context."""
    if ground_truth_context is None:
        # No relevant chunk exists — retrieval 'correctly' returned unrelated chunks
        return None
    return 1.0 if ground_truth_context.strip() in retrieved_context else 0.0

print(f'{'Question':<45} {'GT in Context?':>15}')
print('-' * 62)

precision_scores = []
for r in results:
    score = context_precision(r['retrieved_context'], r['ground_truth_context'])
    q = r['question'][:43] + '..' if len(r['question']) > 43 else r['question']
    label = 'N/A (no GT)' if score is None else ('✓ Yes' if score == 1.0 else '✗ No')
    print(f'{q:<45} {label:>15}')
    if score is not None:
        precision_scores.append(score)

print('-' * 62)
avg = sum(precision_scores) / len(precision_scores)
print(f'Context Precision (avg, excl. N/A): {avg:.2f}')

Question                                       GT in Context?
--------------------------------------------------------------
How many annual leave days do employees get..           ✓ Yes
How many sick leave days are allowed?                   ✓ Yes
How many days can I work from home per week..           ✓ Yes
What is the overtime pay rate?                          ✓ Yes
What is the company stock price?                  N/A (no GT)
--------------------------------------------------------------
Context Precision (avg, excl. N/A): 1.00


## Step 6 — Full Evaluation Report

In [9]:
print('=' * 50)
print('        RAG EVALUATION REPORT')
print('=' * 50)
print(f'  Total test questions : {len(results)}')
print()
print(f'  Faithfulness         : {avg_faith:.2f} / 1.00')
print(f'  Answer Relevance     : {avg_rel:.2f} / 1.00')
print(f'  Context Recall       : {avg_recall:.2f} / 1.00')
print(f'  Context Precision    : {avg:.2f} / 1.00')
print()
overall = (avg_faith + avg_rel + avg_recall + avg) / 4
print(f'  Overall Score        : {overall:.2f} / 1.00')
print('=' * 50)

# Interpretation
print()
print('Interpretation:')
for metric, score in [('Faithfulness', avg_faith), ('Answer Relevance', avg_rel),
                       ('Context Recall', avg_recall), ('Context Precision', avg)]:
    if score >= 0.8:
        status = '✓ Good'
    elif score >= 0.6:
        status = '~ Acceptable'
    else:
        status = '✗ Needs improvement'
    print(f'  {metric:<20} {score:.2f}  {status}')

        RAG EVALUATION REPORT
  Total test questions : 5

  Faithfulness         : 0.90 / 1.00
  Answer Relevance     : 0.90 / 1.00
  Context Recall       : 0.80 / 1.00
  Context Precision    : 1.00 / 1.00

  Overall Score        : 0.90 / 1.00

Interpretation:
  Faithfulness         0.90  ✓ Good
  Answer Relevance     0.90  ✓ Good
  Context Recall       0.80  ✓ Good
  Context Precision    1.00  ✓ Good


## Summary

| Metric | What it measures | Fix if low |
|--------|-----------------|------------|
| **Faithfulness** | Answer grounded in context? | Improve prompt, reduce temperature |
| **Answer Relevance** | Answer addresses the question? | Better prompt, clearer instructions |
| **Context Recall** | Right chunks retrieved? | Tune chunk size, use hybrid search |
| **Context Precision** | No irrelevant chunks? | Reduce `k`, use reranking |

**Evaluation loop:**
```
Build RAG → Evaluate → Identify weak metric → Fix → Re-evaluate
```

For production, run evaluation after every change to chunking strategy, embedding model, or prompt.